# Entry Inspector

Inspect one benchmark entry and every answer the models gave for it.

Pick the **benchmark**, **language**, **entry number** and **experiments** in the
form below, then press **Show**. For each experiment you get:

1. a small table of the entry from `data/benchmarks/`, showing only the
   ground-truth functions;
2. one table of the answers from `results_robustness/` (`old/` is skipped), with
   the experiment and model on every row.

Use **Turns** to switch between all of a model's turns and only its first or final
one. A run can be solved overall even though its earlier turns are invalid, so
*Solved* describes the run and *Turn valid* describes the single turn.

**Without the form** (if widgets don't render in your editor), call the function
directly in a new cell:

```python
inspect_entry("bfcl_multiple", "he", 2)
inspect_entry("bfcl_multiple", "he", 2,
              experiments=["he_translatable_query", "he_translatable_full"],
              models=["claude-opus-5", "grok-4.6"],
              turns="final")
```

In [9]:
# ── Cell 1: Setup ────────────────────────────────────────────────────────────
import html
import json
import re
from functools import lru_cache
from pathlib import Path

from IPython.display import display, HTML

NOTEBOOK_DIR = Path().resolve()
PACKAGE_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
DATA_ROOT    = PACKAGE_ROOT / "data" / "benchmarks"
RESULTS_ROOT = PACKAGE_ROOT / "results_robustness"

# Folders under results_robustness/ that don't hold current model results.
EXCLUDED_RESULT_DIRS = {"old"}

print(f"Package root: {PACKAGE_ROOT}")

Package root: C:\Users\omnoy\Documents\BIU\Thesis\multilingual-tool-use-evaluation\multilingual-bfcl


In [10]:
# ── Cell 2: Load benchmarks + results ────────────────────────────────────────
def list_benchmarks():
    return sorted(d.name for d in DATA_ROOT.iterdir() if d.is_dir())

def list_languages(benchmark):
    return sorted(d.name for d in (DATA_ROOT / benchmark).iterdir() if d.is_dir())

def list_experiments(benchmark, language):
    return sorted(p.stem for p in (DATA_ROOT / benchmark / language).glob("*.jsonl"))

def list_models():
    return sorted(d.name for d in RESULTS_ROOT.iterdir()
                  if d.is_dir() and d.name not in EXCLUDED_RESULT_DIRS)

def transcripts_dir(model, benchmark, language, experiment):
    return RESULTS_ROOT / model / benchmark / language / experiment / "transcripts"

def models_with_results(benchmark, language, experiment):
    return [m for m in list_models()
            if transcripts_dir(m, benchmark, language, experiment).is_dir()]

@lru_cache(maxsize=None)
def load_benchmark(benchmark, language, experiment):
    """Entries of one benchmark file keyed by id, or None if it isn't valid JSONL."""
    path = DATA_ROOT / benchmark / language / f"{experiment}.jsonl"
    try:
        with open(path, encoding="utf-8") as f:
            rows = [json.loads(line) for line in f if line.strip()]
    except json.JSONDecodeError:
        return None
    return {r["id"]: r for r in rows}

def entry_number(entry_id):
    """'multiple_2_he' -> 2, 'wild_tool_bench_0_zh' -> 0 (last integer in the id)."""
    return int(re.findall(r"\d+", entry_id)[-1])

def find_entry(benchmark, language, experiment, number):
    entries = load_benchmark(benchmark, language, experiment) or {}
    for entry_id, entry in entries.items():
        if entry_number(entry_id) == number:
            return entry
    return None

def list_entry_numbers(benchmark, language, experiments):
    numbers = set()
    for experiment in experiments:
        entries = load_benchmark(benchmark, language, experiment) or {}
        numbers.update(entry_number(entry_id) for entry_id in entries)
    return sorted(numbers)

def with_ground_truth_functions_only(entry):
    """Copy of the entry whose `function` list keeps only ground-truth functions."""
    gt_names = {name for call in entry.get("ground_truth", []) for name in call}
    return {**entry, "function": [f for f in entry.get("function", []) if f["name"] in gt_names]}

def load_transcript(model, benchmark, language, experiment, entry_id):
    path = transcripts_dir(model, benchmark, language, experiment) / f"{entry_id}.json"
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return json.load(f)

print(f"Benchmarks: {list_benchmarks()}")
print(f"Models    : {list_models()}")

Benchmarks: ['bfcl_multiple', 'wildtoolbench']
Models    : ['Kimi-K2.6', 'claude-opus-5', 'claude-opus-5-5', 'claude-sonnet-5', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'grok-4.6']


In [11]:
# ── Cell 3: Render ───────────────────────────────────────────────────────────
CARD_BG     = "#1e1e1e"
CARD_BORDER = "#3a3a3a"
SECTION_BG  = "#2a2a2a"
TITLE_COLOR = "#e0e0e0"
LABEL_COLOR = "#aaaaaa"
TEXT_COLOR  = "#d4d4d4"
GOOD_COLOR  = "#6fcf97"
BAD_COLOR   = "#eb7a7a"
MUTED_COLOR = "#888888"

# Model text longer than this starts collapsed.
TEXT_PREVIEW_CHARS = 200

CELL_STYLE = (f"padding:6px 8px;border-top:1px solid {CARD_BORDER};vertical-align:top;"
              f"text-align:left;color:{TEXT_COLOR};background:{CARD_BG}")
HEAD_STYLE = (f"padding:6px 8px;text-align:left;color:{LABEL_COLOR};background:{SECTION_BG};"
              "font-weight:600;white-space:nowrap")
# Heavier rule where a new run (experiment + model) starts.
RUN_START_STYLE = "border-top:2px solid #666666"

def esc(value):
    return html.escape(str(value))

def yes_no(value):
    color, label = (GOOD_COLOR, "yes") if value else (BAD_COLOR, "no")
    return f'<span style="color:{color};font-weight:600">{label}</span>'

def format_calls(tool_calls):
    if not tool_calls:
        return f'<i style="color:{MUTED_COLOR}">no function call</i>'
    lines = []
    for call in tool_calls:
        args = call.get("arguments")
        if isinstance(args, dict):
            args = ", ".join(f"{k}={json.dumps(v, ensure_ascii=False)}" for k, v in args.items())
        lines.append(f'<div dir="ltr"><code style="color:#dcdcaa">{esc(call.get("name"))}</code>'
                     f'(<code style="color:#b5cea8">{esc(args)}</code>)</div>')
    return "".join(lines)

def format_text(texts):
    text = "\n".join(t for t in texts or [] if t.strip())
    if not text:
        return ""
    style = "white-space:pre-wrap;unicode-bidi:plaintext"
    if len(text) <= TEXT_PREVIEW_CHARS:
        return f'<div dir="auto" style="{style}">{esc(text)}</div>'
    return (f'<details><summary dir="auto" style="cursor:pointer">{esc(text[:TEXT_PREVIEW_CHARS])}…</summary>'
            f'<div dir="auto" style="{style};margin-top:4px">{esc(text)}</div></details>')

def format_error(attempt):
    if not attempt.get("error_type"):
        return ""
    title = attempt.get("checker_error")
    title_attr = f' title="{esc(json.dumps(title, ensure_ascii=False))}"' if title else ""
    return f'<code{title_attr} style="color:{LABEL_COLOR}">{esc(attempt["error_type"])}</code>'

def select_turns(attempts, turns):
    if turns == "first":
        return attempts[:1]
    if turns == "final":
        return attempts[-1:]
    return attempts

def collect_answer_rows(benchmark, language, experiment, number, models, turns):
    """Table rows for one experiment, plus the models that have no answer for it."""
    entry = find_entry(benchmark, language, experiment, number)
    rows, missing = [], []
    if entry is None:
        return rows, missing
    for model in models:
        transcript = load_transcript(model, benchmark, language, experiment, entry["id"])
        if transcript is None:
            missing.append(model)
            continue
        attempts = transcript.get("attempts") or []
        run = {"experiment": experiment, "model": model, "solved": transcript.get("success"),
               "num_turns": len(attempts), "run_error": transcript.get("error")}
        if not attempts:
            rows.append({**run, "attempt": None, "run_start": True})
            continue
        for i, attempt in enumerate(select_turns(attempts, turns)):
            rows.append({**run, "attempt": attempt, "run_start": i == 0})
    return rows, missing

def render_row(row):
    attempt = row["attempt"]
    if attempt is None:
        turn = valid = error = text = ""
        answer = f'<span style="color:{BAD_COLOR}">run failed: {esc(row["run_error"])}</span>'
    else:
        turn   = f'{attempt["attempt"]} / {row["num_turns"]}'
        valid  = yes_no(attempt.get("valid"))
        answer = format_calls(attempt.get("tool_calls"))
        error  = format_error(attempt)
        text   = format_text(attempt.get("text"))
    cells = [f"<code>{esc(row['experiment'])}</code>", f"<b>{esc(row['model'])}</b>",
             yes_no(row["solved"]), turn, valid, answer, error, text]
    extra = RUN_START_STYLE if row["run_start"] else ""
    # The short columns stay on one line so the answer and text get the remaining width.
    return "<tr>" + "".join(
        f'<td style="{CELL_STYLE};{extra};{"white-space:nowrap" if i < 5 else ""}">{c}</td>'
        for i, c in enumerate(cells)) + "</tr>"

def render_answers_table(rows):
    if not rows:
        return f'<i style="color:{MUTED_COLOR}">No answers found in results_robustness/ for this selection.</i>'
    headers = ["Experiment", "Model", "Solved", "Turn", "Turn valid", "Answer", "Error", "Model text"]
    head = "".join(f'<th style="{HEAD_STYLE}">{h}</th>' for h in headers)
    body = "".join(render_row(r) for r in rows)
    return f'<table style="border-collapse:collapse;width:100%;font-size:0.88em"><tr>{head}</tr>{body}</table>'

def format_query(entry):
    parts = []
    for turn in entry.get("messages", []):
        for message in turn:
            parts.append(f'<div><span style="color:{LABEL_COLOR}">{esc(message.get("role"))}:</span> '
                         f'<span dir="auto" style="white-space:pre-wrap">{esc(message.get("content"))}</span></div>')
    return "".join(parts)

def format_function(function):
    parts = [f'<div><code style="color:#dcdcaa">{esc(function["name"])}</code></div>',
             f'<div dir="auto" style="color:{LABEL_COLOR}">{esc(function.get("description", ""))}</div>']
    parameters = function.get("parameters", {})
    required = parameters.get("required", [])
    for name, spec in parameters.get("properties", {}).items():
        details = [esc(spec.get("type", "any"))]
        if spec.get("items"):
            details[0] += " of " + esc(spec["items"].get("type", "any"))
        if spec.get("properties"):
            details.append("fields " + esc(", ".join(spec["properties"])))
        if spec.get("enum"):
            details.append("one of " + esc(" | ".join(map(str, spec["enum"]))))
        if "default" in spec:
            details.append("default " + esc(json.dumps(spec["default"], ensure_ascii=False)))
        star = "*" if name in required else ""
        parts.append(f'<div style="margin-top:2px"><code style="color:#ce9178">{esc(name)}{star}</code> '
                     f'<span style="color:{MUTED_COLOR}">{", ".join(details)}</span> '
                     f'<span dir="auto">{esc(spec.get("description", ""))}</span></div>')
    return "".join(parts)

def format_ground_truth(ground_truth):
    parts = []
    for call in ground_truth:
        for name, params in call.items():
            parts.append(f'<div><code style="color:#dcdcaa">{esc(name)}</code></div>')
            for param, values in params.items():
                shown = f' <span style="color:{MUTED_COLOR}">|</span> '.join(
                    f'<code style="color:#b5cea8">{esc(json.dumps(v, ensure_ascii=False))}</code>' for v in values)
                parts.append(f'<div style="margin-top:2px"><code style="color:#ce9178">{esc(param)}</code> = {shown}</div>')
    return "".join(parts)

def render_entry_table(entry):
    """Small label/value table of an entry, with only its ground-truth functions."""
    shown = with_ground_truth_functions_only(entry)
    hidden = len(entry.get("function", [])) - len(shown["function"])
    identifiers = [entry.get(k) for k in ("id", "source_id", "locale", "localization_level") if entry.get(k)]
    function_label = f'Function<div style="color:{MUTED_COLOR};font-weight:400">{hidden} others hidden</div>'
    rows = [("Entry", esc(" · ".join(identifiers))),
            ("Query", format_query(entry)),
            (function_label, "".join(format_function(f) for f in shown["function"])),
            ("Ground truth", format_ground_truth(entry.get("ground_truth", [])))]
    body = "".join(f'<tr><td style="{CELL_STYLE};color:{LABEL_COLOR};font-weight:600;white-space:nowrap">{label}</td>'
                   f'<td style="{CELL_STYLE};overflow-wrap:anywhere">{value}</td></tr>' for label, value in rows)
    return f'<table style="border-collapse:collapse;width:100%;font-size:0.85em">{body}</table>'

def render_entry_card(benchmark, language, experiment, number):
    if load_benchmark(benchmark, language, experiment) is None:
        body = f'<i style="color:{BAD_COLOR}">This file is not valid JSONL, so it could not be read.</i>'
    else:
        entry = find_entry(benchmark, language, experiment, number)
        if entry is None:
            body = f'<i style="color:{MUTED_COLOR}">No entry {number} in this file.</i>'
        else:
            body = render_entry_table(entry)
    return (f'<div style="min-width:0"><div style="font-weight:600;color:{LABEL_COLOR};margin-bottom:4px">'
            f'{esc(experiment)}</div>{body}</div>')

def render_inspection(benchmark, language, number, experiments, models, turns):
    cards = "".join(render_entry_card(benchmark, language, e, number) for e in experiments)
    rows, missing_notes = [], []
    for experiment in experiments:
        experiment_rows, missing = collect_answer_rows(benchmark, language, experiment, number, models, turns)
        rows.extend(experiment_rows)
        if missing:
            missing_notes.append(f"<code>{esc(experiment)}</code>: {esc(', '.join(missing))}")
    notes = ""
    if missing_notes:
        notes = (f'<div style="color:{MUTED_COLOR};font-size:0.85em;margin-top:8px">'
                 f'No results for: {"; ".join(missing_notes)}</div>')
    section = f'font-size:1.05em;font-weight:700;color:{TITLE_COLOR};margin:0 0 10px'
    return f"""
<div style="border:1px solid {CARD_BORDER};border-radius:8px;padding:16px;
            font-family:sans-serif;background:{CARD_BG};color:{TEXT_COLOR}">
  <div style="{section}">{esc(benchmark)} / {esc(language)} / entry {number}</div>
  <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(340px,1fr));gap:12px;margin-bottom:18px">{cards}</div>
  <div style="{section}">Answers ({len(rows)} rows)</div>
  <div style="overflow-x:auto">{render_answers_table(rows)}</div>
  {notes}
</div>"""

def inspect_entry(benchmark, language, number, experiments=None, models=None, turns="all"):
    """Display one entry. experiments/models default to everything with results;
    turns is 'all', 'first' or 'final'."""
    if experiments is None:
        all_experiments = list_experiments(benchmark, language)
        experiments = ([e for e in all_experiments if models_with_results(benchmark, language, e)]
                       or all_experiments)
    display(HTML(render_inspection(benchmark, language, number, experiments,
                                   models or list_models(), turns)))

In [12]:
# ── Cell 4: Form ─────────────────────────────────────────────────────────────
import ipywidgets as widgets

label_layout = {"description_width": "90px"}

class DropdownChecklist:
    """Multi-select dropdown: a button that opens a checklist (ipywidgets has none built in).
    `options` takes (label, value) pairs and `value` is the tuple of checked values."""

    def __init__(self, title, width):
        self.title = title
        self.toggle = widgets.Button(layout=widgets.Layout(width=width))
        self.toggle.on_click(self._on_toggle)
        select_all = widgets.Button(description="All", layout=widgets.Layout(width="60px"))
        select_none = widgets.Button(description="None", layout=widgets.Layout(width="60px"))
        select_all.on_click(lambda _: self._check(lambda option_value: True))
        select_none.on_click(lambda _: self._check(lambda option_value: False))
        self.checklist = widgets.VBox(layout=widgets.Layout(max_height="220px", overflow="auto"))
        self.panel = widgets.VBox([widgets.HBox([select_all, select_none]), self.checklist],
                                  layout=widgets.Layout(display="none", width=width, border="1px solid #888888",
                                                        padding="4px"))
        self.widget = widgets.VBox([self.toggle, self.panel])
        self.options = []

    @property
    def options(self):
        return self._options

    @options.setter
    def options(self, options):
        self._options = list(options)
        self._boxes = {value: widgets.Checkbox(description=label, indent=False, value=False,
                                               layout=widgets.Layout(width="auto"))
                       for label, value in self._options}
        for box in self._boxes.values():
            box.observe(self._update_title, names="value")
        self.checklist.children = tuple(self._boxes.values())
        self._update_title()

    @property
    def value(self):
        return tuple(value for value, box in self._boxes.items() if box.value)

    @value.setter
    def value(self, values):
        self._check(lambda option_value: option_value in values)

    def _check(self, is_checked):
        for value, box in self._boxes.items():
            box.value = is_checked(value)

    def _on_toggle(self, _):
        is_open = self.panel.layout.display != "none"
        self.panel.layout.display = "none" if is_open else "flex"
        self._update_title()

    def _update_title(self, *_):
        arrow = "▲" if self.panel.layout.display != "none" else "▼"
        self.toggle.description = f"{self.title}: {len(self.value)} of {len(self._boxes)} {arrow}"

benchmark_input = widgets.Dropdown(description="Benchmark", options=list_benchmarks(),
                                    style=label_layout, layout=widgets.Layout(width="320px"))
language_input   = widgets.Dropdown(description="Language", style=label_layout,
                                    layout=widgets.Layout(width="320px"))
number_input     = widgets.Dropdown(description="Entry number", style=label_layout,
                                    layout=widgets.Layout(width="210px"))
previous_button  = widgets.Button(description="◀", tooltip="Previous entry", layout=widgets.Layout(width="50px"))
next_button      = widgets.Button(description="▶", tooltip="Next entry", layout=widgets.Layout(width="50px"))
turns_input      = widgets.Dropdown(description="Turns", style=label_layout,
                                    options=[("All turns", "all"), ("First turn", "first"), ("Final turn", "final")],
                                    layout=widgets.Layout(width="320px"))
experiments_input = DropdownChecklist("Experiments", width="420px")
models_input      = DropdownChecklist("Models", width="320px")
show_button = widgets.Button(description="Show", button_style="primary")
output = widgets.Output()

def experiment_label(benchmark, language, experiment):
    if load_benchmark(benchmark, language, experiment) is None:
        return f"{experiment} (unreadable)"
    count = len(models_with_results(benchmark, language, experiment))
    return f"{experiment} ({count} models)" if count else f"{experiment} (no results)"

def refresh_languages(*_):
    previous_language = language_input.value
    languages = list_languages(benchmark_input.value)
    language_input.options = languages
    language_input.value = previous_language if previous_language in languages else languages[0]
    # The language may be unchanged across benchmarks, which fires no observer.
    refresh_selectors()

def refresh_selectors(*_):
    benchmark, language = benchmark_input.value, language_input.value
    if language is None:
        return
    experiments = list_experiments(benchmark, language)
    with_results = [e for e in experiments if models_with_results(benchmark, language, e)]
    experiments_input.options = [(experiment_label(benchmark, language, e), e) for e in experiments]
    experiments_input.value = tuple(with_results or experiments)

    models = sorted({m for e in with_results for m in models_with_results(benchmark, language, e)})
    models_input.options = [(m, m) for m in models]
    models_input.value = tuple(models)

    previous_number = number_input.value
    numbers = list_entry_numbers(benchmark, language, experiments)
    number_input.options = numbers
    number_input.value = (previous_number if previous_number in numbers
                          else numbers[0] if numbers else None)

def show(*_):
    with output:
        output.clear_output(wait=True)
        if not experiments_input.value or number_input.value is None:
            print("Select at least one experiment and an entry number.")
            return
        inspect_entry(benchmark_input.value, language_input.value, number_input.value,
                      experiments=list(experiments_input.value),
                      models=list(models_input.value), turns=turns_input.value)

def step(offset):
    def on_click(_):
        numbers = list(number_input.options)
        if number_input.value is None:
            return
        index = numbers.index(number_input.value) + offset
        if 0 <= index < len(numbers):
            number_input.value = numbers[index]
            show()
    return on_click

benchmark_input.observe(refresh_languages, names="value")
language_input.observe(refresh_selectors, names="value")
previous_button.on_click(step(-1))
next_button.on_click(step(1))
show_button.on_click(show)

refresh_languages()

display(widgets.VBox([
    widgets.HBox([benchmark_input, language_input]),
    widgets.HBox([experiments_input.widget, models_input.widget],
                 layout=widgets.Layout(align_items="flex-start")),
    widgets.HBox([number_input, previous_button, next_button, turns_input, show_button]),
    output,
]))
show()